# TP 8 — Fausses couleurs, probabilités et calibration

[![Ouvrir dans Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/earthai-tech/lab-courses/blob/main/curat/data-science/notebooks/TP8_Incertitude_calibration.ipynb)

**Data Sciences, visualisation et Big Data** · M2 Télédétection et SIG ·
Séance 8 · KOUADIO Kouao Laurent

Même zone et mêmes données qu'au TP 5 (Yamoussoukro, 60 m).
Objectifs :
1. Composer et **étirer** des images en vraies et fausses couleurs.
2. Cartographier la **confiance** et l'**entropie** d'une forêt
   aléatoire.
3. Vérifier la **calibration** (diagramme de fiabilité, ECE).
4. Utiliser l'incertitude pour **décider** où vérifier.

Durée : environ 90 minutes. La lecture des images prend environ
6 minutes : lancez la partie 1 dès le début.

In [ ]:
# Google Colab : installe les paquets du cours (environ 1 minute).
# Dans l'environnement conda du cours (environment.yml), rien à faire.
import sys
if "google.colab" in sys.modules:
    %pip install -q pystac-client odc-stac rioxarray k-diagram

## 1. Données (comme au TP 5)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import rioxarray
import xarray as xr
from kdiagram.plot.comparison import (plot_polar_reliability,
                                      plot_reliability_diagram)
from kdiagram.plot.evaluation import plot_polar_confusion_multiclass
from matplotlib.colors import ListedColormap
from odc.stac import configure_rio, load
from pystac_client import Client
from rasterio.enums import Resampling
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupKFold, cross_val_predict

configure_rio(cloud_defaults=True, GDAL_HTTP_MAX_RETRY="10",
              GDAL_HTTP_RETRY_DELAY="2")
GRAINE = 0
BBOX = [-5.40, 6.70, -5.10, 7.00]
CRS, RES = "EPSG:32630", 60
BANDES = ["blue", "green", "red", "nir", "swir16", "swir22"]
CLASSES = {10: ("Arbres", "#006400"), 20: ("Arbustes", "#ffbb22"),
           30: ("Herbacées", "#ffff4c"), 40: ("Cultures", "#f096ff"),
           50: ("Bâti", "#fa0000"), 60: ("Sol nu", "#b4b4b4"),
           80: ("Eau", "#0064c8"), 90: ("Zones humides", "#0096a0")}

cat = Client.open("https://earth-search.aws.element84.com/v1")
items = cat.search(collections=["sentinel-2-l2a"], bbox=BBOX,
                   datetime="2020-12-01/2021-02-28",
                   query={"eo:cloud_cover": {"lt": 40}}
                   ).item_collection()
ds = load(items, bands=BANDES + ["scl"], bbox=BBOX, crs=CRS,
          resolution=RES, chunks={}, groupby="solar_day")
clair = ds["scl"].isin([4, 5, 6, 11])
comp = xr.Dataset({b: ds[b].where(clair) * 1e-4 for b in BANDES})
comp = comp.median("time").compute()
B = {b: comp[b].values for b in BANDES}

url = ("https://esa-worldcover.s3.eu-central-1.amazonaws.com/v200/"
       "2021/map/ESA_WorldCover_10m_2021_v200_N06W006_Map.tif")
wc = rioxarray.open_rasterio(url).squeeze(drop=True)
wc = wc.rio.clip_box(*BBOX, crs="EPSG:4326")
lab = wc.rio.reproject_match(comp["red"].rio.write_crs(CRS),
                             resampling=Resampling.mode).values
H, W = lab.shape
print(len(items), "scènes ; grille", H, "x", W)

## 2. Compositions colorées et étirement

In [ ]:
def etirer(a, bas=2, haut=98):
    p1, p2 = np.nanpercentile(a, [bas, haut])
    return np.clip((a - p1) / (p2 - p1), 0, 1)


def rgb(r, v, b, bas=2, haut=98):
    out = np.stack([etirer(B[k], bas, haut) for k in (r, v, b)], -1)
    return np.where(np.isfinite(out), out, 1)


fig, axs = plt.subplots(1, 3, figsize=(15, 5))
for ax, (t, c) in zip(axs, [("naturelles", ("red", "green", "blue")),
                            ("fausses couleurs", ("nir", "red",
                                                  "green")),
                            ("SWIR/PIR/rouge", ("swir22", "nir",
                                                "red"))]):
    ax.imshow(rgb(*c))
    ax.set_title(t)
    ax.axis("off")
plt.show()

fig, axs = plt.subplots(1, 2, figsize=(10, 5))
axs[0].imshow(rgb("red", "green", "blue", 0, 100))
axs[0].set_title("min–max (0–100 %)")
axs[1].imshow(rgb("red", "green", "blue", 2, 98))
axs[1].set_title("2–98 %")
for ax in axs:
    ax.axis("off")
plt.show()

**Question 1.** Dans les fausses couleurs, pourquoi la végétation
est-elle rouge ? Pourquoi l'étirement min–max donne-t-il une image
sombre ? Tracez l'histogramme de la bande rouge pour justifier.

## 3. Probabilités par validation spatiale

In [ ]:
F = dict(B)
F["NDVI"] = (B["nir"] - B["red"]) / (B["nir"] + B["red"])
F["NDWI"] = (B["green"] - B["nir"]) / (B["green"] + B["nir"])
F["NDBI"] = (B["swir16"] - B["nir"]) / (B["swir16"] + B["nir"])
X_tout = np.stack(list(F.values()), -1)
valide = np.isfinite(X_tout).all(-1) & np.isin(lab, list(CLASSES))

rng = np.random.default_rng(GRAINE)
garder = [c for c in CLASSES if (lab[valide] == c).sum() >= 300]
idx = np.concatenate([
    rng.choice(np.flatnonzero((lab == c).ravel() & valide.ravel()),
               size=min(300, int(((lab == c) & valide).sum())),
               replace=False) for c in garder])
X = X_tout.reshape(-1, X_tout.shape[-1])[idx]
y = lab.ravel()[idx]
lig, col = np.unravel_index(idx, (H, W))
groupes = (lig // 50) * 1000 + col // 50

rf = RandomForestClassifier(n_estimators=300, n_jobs=-1,
                            random_state=GRAINE)
P = cross_val_predict(rf, X, y, cv=GroupKFold(5), groups=groupes,
                      method="predict_proba")
classes = np.unique(y)
pred = classes[P.argmax(1)]
conf = P.max(1)
juste = (pred == y).astype(int)
print(f"précision : {juste.mean():.3f}  confiance moyenne : "
      f"{conf.mean():.3f}")

**Question 2.** Le modèle est-il, en moyenne, trop sûr de lui ou
pas assez ?

## 4. Calibration

In [ ]:
plot_reliability_diagram(juste, conf, names=["forêt aléatoire"],
                         n_bins=10, figsize=(7, 7))
plt.show()

bins = np.clip((conf * 10).astype(int), 0, 9)
ece = sum(abs(juste[bins == k].mean() - conf[bins == k].mean())
          * (bins == k).mean() for k in range(10) if (bins == k).any())
print(f"ECE = {ece:.3f}")

In [ ]:
plot_polar_reliability(juste, conf, names=["forêt aléatoire"],
                       n_bins=10, figsize=(6, 6))
plt.show()
plot_polar_confusion_multiclass(
    y, pred, class_labels=[CLASSES[c][0] for c in classes],
    figsize=(7, 7))
plt.show()

**Question 3.** Écrivez la formule de l'ECE que le code calcule.
Dans quelles tranches de confiance le modèle se trompe-t-il le plus
sur sa propre fiabilité ?

## 5. Cartes de confiance et d'entropie

In [ ]:
rf.fit(X, y)
plat = X_tout.reshape(-1, X_tout.shape[-1])
vv = valide.ravel()
Pm = np.full((H * W, len(classes)), np.nan)
Pm[vv] = rf.predict_proba(plat[vv])
cmax = np.nanmax(Pm, 1)
ent = -np.nansum(Pm * np.log(Pm + 1e-12), 1) / np.log(len(classes))
classe = np.full(H * W, np.nan)
classe[vv] = Pm[vv].argmax(1)
cmax[~vv] = np.nan
ent[~vv] = np.nan

cmap = ListedColormap([CLASSES[c][1] for c in classes])
fig, axs = plt.subplots(1, 3, figsize=(16, 5.5))
axs[0].imshow(classe.reshape(H, W), cmap=cmap, vmin=0,
              vmax=len(classes) - 1, interpolation="nearest")
axs[0].set_title("classe la plus probable")
im = axs[1].imshow(cmax.reshape(H, W), cmap="viridis", vmin=0.2,
                   vmax=1)
axs[1].set_title("confiance")
fig.colorbar(im, ax=axs[1], shrink=0.7)
im = axs[2].imshow(ent.reshape(H, W), cmap="magma", vmin=0, vmax=1)
axs[2].set_title("entropie normalisée")
fig.colorbar(im, ax=axs[2], shrink=0.7)
for ax in axs:
    ax.axis("off")
plt.show()

## 6. Décider où vérifier

In [ ]:
a_verifier = cmax.reshape(H, W) < 0.5
print(f"{100 * np.nanmean(a_verifier[valide]):.1f} % de la zone "
      "à vérifier")
gris = rgb("red", "green", "blue").mean(-1)
plt.figure(figsize=(6, 6))
plt.imshow(gris, cmap="gray", vmin=0, vmax=1.2)
calque = np.zeros((H, W, 4))
calque[a_verifier] = [0.88, 0.48, 0.12, 0.7]
plt.imshow(calque)
plt.title("confiance < 0,5")
plt.axis("off")
plt.show()

**Question 4.** Rédigez une note de **5 lignes** pour un service
d'urbanisme : que peut-on affirmer avec cette carte, où, et avec
quelle confiance ? Que faut-il vérifier sur le terrain ?

**Question 5 (bonus).** Recommencez avec `n_estimators=30`. La
calibration et la carte d'entropie changent-elles ?

## À rendre

Le notebook exécuté et la note de la question 4.